In [ ]:
for soybeans may 1st to oct 30

# SAR feature data set only 

In [4]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
DATA_PATH = "data/soybeans_2016_2023_processed.parquet" 
df = pd.read_parquet(DATA_PATH)

# Ensure no NaNs in target
df = df[df["yield"].notna()].copy()
print(f"Loaded {len(df)} rows with {len(df.columns)} columns.")

# ---------------- 2. DYNAMIC FEATURE GROUPING ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("PR_", "RVI_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["DEM", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]

SAR_ALL = SAR_RAW + SAR_INDICES

# ---------------- 3. SPATIAL FEATURE GENERATION ----------------
print("Calculating Field-Level Spatial Features (Means)...")
field_means = df.groupby(['farm_name', 'Year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

SAR_SPATIAL = SAR_ALL + list(field_means.columns)

def add_spatial_neighbor_features(df, sar_cols, n_neighbors=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    
    for farm in df['farm_name'].unique():
        farm_mask = df['farm_name'] == farm
        farm_df = df[farm_mask]
        if len(farm_df) <= n_neighbors: continue
            
        tree = BallTree(farm_df[['latitude', 'longitude']].values)
        _, indices = tree.query(farm_df[['latitude', 'longitude']].values, k=n_neighbors+1)
        
        for i, col in enumerate(sar_cols):
            vals = farm_df[col].values
            neighbor_data[farm_mask, i] = vals[indices[:, 1:]].mean(axis=1)
            
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbor_features(df, SAR_ALL)

# ---------------- 4. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "BASE" 

FEATURE_MAP = {
    "BASE":         SAR_ALL + ["Year", "latitude"],
    "WEATHER":      SAR_ALL + WEATHER + ["Year", "latitude"],
    "TERRAIN":      SAR_ALL + WEATHER + TERRAIN + SOIL + ["Year", "latitude"],
    "FULL_SPATIAL": SAR_SPATIAL + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["Year", "latitude", "longitude"]
}

SELECTED_FEATURES = FEATURE_MAP[RUN_MODE]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["Year"]

# Stratification for balanced folds
y_binned = pd.qcut(y, q=5, labels=False)

print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: H200 GPU")

# ---------------- 5. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]
    
    # Year-Mean Normalization
    yr_map = df.iloc[tr].groupby("Year")["yield"].mean().to_dict()
    global_mean = y_tr.mean()
    mu_tr = years.iloc[tr].map(yr_map).fillna(global_mean).values
    mu_te = years.iloc[te].map(yr_map).fillna(global_mean).values

    # Model Definition (with Early Stopping in constructor for XGB 2.0+)
    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        reg_alpha=1.0,
        early_stopping_rounds=50
    )
    
    model.fit(X_tr, y_tr / mu_tr, eval_set=[(X_te, y_te / mu_te)], verbose=False)
    
    p_ratio = model.predict(X_te)
    p_raw = p_ratio * mu_te

    # --- METRIC CALCULATIONS ---
    r2 = r2_score(y_te, p_raw)
    mae = mean_absolute_error(y_te, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te, p_raw))
    
    y_range = np.max(y_te) - np.min(y_te)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    mape = mean_absolute_percentage_error(y_te, p_raw) * 100
    
    z_true = np.where(y_te/mu_te < 0.9, 0, np.where(y_te/mu_te <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        'R2': r2, 'MAE': mae, 'RMSE': rmse, 
        'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc
    })

    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 6. DETAILED SUMMARY ----------------
res_df = pd.DataFrame(results)

print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 694250 rows with 323 columns.
Calculating Field-Level Spatial Features (Means)...
Generating BallTree neighbor features for 264 features...
🚀 Mode: BASE | Features: 266 | Device: H200 GPU
Fold 1 | R2: 0.521 | RMSE: 9.32 | MAPE: 16.10% | Time: 3.1s
Fold 2 | R2: 0.492 | RMSE: 10.42 | MAPE: 22.47% | Time: 3.1s
Fold 3 | R2: 0.559 | RMSE: 9.08 | MAPE: 16.23% | Time: 3.2s
Fold 4 | R2: 0.403 | RMSE: 10.43 | MAPE: 19.96% | Time: 5.1s
Fold 5 | R2: 0.306 | RMSE: 11.26 | MAPE: 19.20% | Time: 2.5s

📊 FINAL SUMMARY: BASE
Total Time: 17.86s
-----------------------------------
R²:    0.4560 ± 0.1018
MAE:   7.71
RMSE:  10.10
NRMSE: 10.70%
MAPE:  18.79%
Acc:   0.5138
-----------------------------------


# SAR feature data set with weather 

In [5]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
DATA_PATH = "data/soybeans_2016_2023_processed.parquet" 
df = pd.read_parquet(DATA_PATH)

# Ensure no NaNs in target
df = df[df["yield"].notna()].copy()
print(f"Loaded {len(df)} rows with {len(df.columns)} columns.")

# ---------------- 2. DYNAMIC FEATURE GROUPING ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("PR_", "RVI_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["DEM", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]

SAR_ALL = SAR_RAW + SAR_INDICES

# ---------------- 3. SPATIAL FEATURE GENERATION ----------------
print("Calculating Field-Level Spatial Features (Means)...")
field_means = df.groupby(['farm_name', 'Year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

SAR_SPATIAL = SAR_ALL + list(field_means.columns)

def add_spatial_neighbor_features(df, sar_cols, n_neighbors=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    
    for farm in df['farm_name'].unique():
        farm_mask = df['farm_name'] == farm
        farm_df = df[farm_mask]
        if len(farm_df) <= n_neighbors: continue
            
        tree = BallTree(farm_df[['latitude', 'longitude']].values)
        _, indices = tree.query(farm_df[['latitude', 'longitude']].values, k=n_neighbors+1)
        
        for i, col in enumerate(sar_cols):
            vals = farm_df[col].values
            neighbor_data[farm_mask, i] = vals[indices[:, 1:]].mean(axis=1)
            
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbor_features(df, SAR_ALL)

# ---------------- 4. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "WEATHER" 

FEATURE_MAP = {
    "BASE":         SAR_ALL + ["Year", "latitude"],
    "WEATHER":      SAR_ALL + WEATHER + ["Year", "latitude"],
    "TERRAIN":      SAR_ALL + WEATHER + TERRAIN + SOIL + ["Year", "latitude"],
    "FULL_SPATIAL": SAR_SPATIAL + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["Year", "latitude", "longitude"]
}

SELECTED_FEATURES = FEATURE_MAP[RUN_MODE]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["Year"]

# Stratification for balanced folds
y_binned = pd.qcut(y, q=5, labels=False)

print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: H200 GPU")

# ---------------- 5. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]
    
    # Year-Mean Normalization
    yr_map = df.iloc[tr].groupby("Year")["yield"].mean().to_dict()
    global_mean = y_tr.mean()
    mu_tr = years.iloc[tr].map(yr_map).fillna(global_mean).values
    mu_te = years.iloc[te].map(yr_map).fillna(global_mean).values

    # Model Definition (with Early Stopping in constructor for XGB 2.0+)
    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        reg_alpha=1.0,
        early_stopping_rounds=50
    )
    
    model.fit(X_tr, y_tr / mu_tr, eval_set=[(X_te, y_te / mu_te)], verbose=False)
    
    p_ratio = model.predict(X_te)
    p_raw = p_ratio * mu_te

    # --- METRIC CALCULATIONS ---
    r2 = r2_score(y_te, p_raw)
    mae = mean_absolute_error(y_te, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te, p_raw))
    
    y_range = np.max(y_te) - np.min(y_te)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    mape = mean_absolute_percentage_error(y_te, p_raw) * 100
    
    z_true = np.where(y_te/mu_te < 0.9, 0, np.where(y_te/mu_te <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        'R2': r2, 'MAE': mae, 'RMSE': rmse, 
        'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc
    })

    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 6. DETAILED SUMMARY ----------------
res_df = pd.DataFrame(results)

print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 694250 rows with 323 columns.
Calculating Field-Level Spatial Features (Means)...
Generating BallTree neighbor features for 264 features...
🚀 Mode: WEATHER | Features: 287 | Device: H200 GPU
Fold 1 | R2: 0.505 | RMSE: 9.48 | MAPE: 16.19% | Time: 2.9s
Fold 2 | R2: 0.407 | RMSE: 11.26 | MAPE: 24.98% | Time: 2.7s
Fold 3 | R2: 0.573 | RMSE: 8.93 | MAPE: 16.72% | Time: 3.5s
Fold 4 | R2: 0.412 | RMSE: 10.35 | MAPE: 19.37% | Time: 5.7s
Fold 5 | R2: 0.233 | RMSE: 11.84 | MAPE: 20.73% | Time: 2.3s

📊 FINAL SUMMARY: WEATHER
Total Time: 17.87s
-----------------------------------
R²:    0.4261 ± 0.1281
MAE:   7.98
RMSE:  10.37
NRMSE: 10.97%
MAPE:  19.60%
Acc:   0.5020
-----------------------------------


# SAR feature data set with weather and terrain features 

In [6]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
DATA_PATH = "data/soybeans_2016_2023_processed.parquet" 
df = pd.read_parquet(DATA_PATH)

# Ensure no NaNs in target
df = df[df["yield"].notna()].copy()
print(f"Loaded {len(df)} rows with {len(df.columns)} columns.")

# ---------------- 2. DYNAMIC FEATURE GROUPING ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("PR_", "RVI_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["DEM", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]

SAR_ALL = SAR_RAW + SAR_INDICES

# ---------------- 3. SPATIAL FEATURE GENERATION ----------------
print("Calculating Field-Level Spatial Features (Means)...")
field_means = df.groupby(['farm_name', 'Year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

SAR_SPATIAL = SAR_ALL + list(field_means.columns)

def add_spatial_neighbor_features(df, sar_cols, n_neighbors=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    
    for farm in df['farm_name'].unique():
        farm_mask = df['farm_name'] == farm
        farm_df = df[farm_mask]
        if len(farm_df) <= n_neighbors: continue
            
        tree = BallTree(farm_df[['latitude', 'longitude']].values)
        _, indices = tree.query(farm_df[['latitude', 'longitude']].values, k=n_neighbors+1)
        
        for i, col in enumerate(sar_cols):
            vals = farm_df[col].values
            neighbor_data[farm_mask, i] = vals[indices[:, 1:]].mean(axis=1)
            
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbor_features(df, SAR_ALL)

# ---------------- 4. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "TERRAIN" 

FEATURE_MAP = {
    "BASE":         SAR_ALL + ["Year", "latitude"],
    "WEATHER":      SAR_ALL + WEATHER + ["Year", "latitude"],
    "TERRAIN":      SAR_ALL + WEATHER + TERRAIN + SOIL + ["Year", "latitude"],
    "FULL_SPATIAL": SAR_SPATIAL + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["Year", "latitude", "longitude"]
}

SELECTED_FEATURES = FEATURE_MAP[RUN_MODE]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["Year"]

# Stratification for balanced folds
y_binned = pd.qcut(y, q=5, labels=False)

print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: H200 GPU")

# ---------------- 5. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]
    
    # Year-Mean Normalization
    yr_map = df.iloc[tr].groupby("Year")["yield"].mean().to_dict()
    global_mean = y_tr.mean()
    mu_tr = years.iloc[tr].map(yr_map).fillna(global_mean).values
    mu_te = years.iloc[te].map(yr_map).fillna(global_mean).values

    # Model Definition (with Early Stopping in constructor for XGB 2.0+)
    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        reg_alpha=1.0,
        early_stopping_rounds=50
    )
    
    model.fit(X_tr, y_tr / mu_tr, eval_set=[(X_te, y_te / mu_te)], verbose=False)
    
    p_ratio = model.predict(X_te)
    p_raw = p_ratio * mu_te

    # --- METRIC CALCULATIONS ---
    r2 = r2_score(y_te, p_raw)
    mae = mean_absolute_error(y_te, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te, p_raw))
    
    y_range = np.max(y_te) - np.min(y_te)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    mape = mean_absolute_percentage_error(y_te, p_raw) * 100
    
    z_true = np.where(y_te/mu_te < 0.9, 0, np.where(y_te/mu_te <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        'R2': r2, 'MAE': mae, 'RMSE': rmse, 
        'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc
    })

    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 6. DETAILED SUMMARY ----------------
res_df = pd.DataFrame(results)

print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 694250 rows with 323 columns.
Calculating Field-Level Spatial Features (Means)...
Generating BallTree neighbor features for 264 features...
🚀 Mode: TERRAIN | Features: 296 | Device: H200 GPU
Fold 1 | R2: 0.525 | RMSE: 9.28 | MAPE: 15.64% | Time: 4.8s
Fold 2 | R2: 0.458 | RMSE: 10.76 | MAPE: 23.62% | Time: 4.7s
Fold 3 | R2: 0.590 | RMSE: 8.75 | MAPE: 16.29% | Time: 3.6s
Fold 4 | R2: 0.377 | RMSE: 10.65 | MAPE: 20.45% | Time: 3.3s
Fold 5 | R2: 0.236 | RMSE: 11.81 | MAPE: 20.27% | Time: 2.5s

📊 FINAL SUMMARY: TERRAIN
Total Time: 19.61s
-----------------------------------
R²:    0.4374 ± 0.1374
MAE:   7.91
RMSE:  10.25
NRMSE: 10.84%
MAPE:  19.25%
Acc:   0.5082
-----------------------------------


# SAR features data set with weather, terrain features, and spatial feature

In [7]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
DATA_PATH = "data/soybeans_2016_2023_processed.parquet" 
df = pd.read_parquet(DATA_PATH)

# Ensure no NaNs in target
df = df[df["yield"].notna()].copy()
print(f"Loaded {len(df)} rows with {len(df.columns)} columns.")

# ---------------- 2. DYNAMIC FEATURE GROUPING ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("PR_", "RVI_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["DEM", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]

SAR_ALL = SAR_RAW + SAR_INDICES

# ---------------- 3. SPATIAL FEATURE GENERATION ----------------
print("Calculating Field-Level Spatial Features (Means)...")
field_means = df.groupby(['farm_name', 'Year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

SAR_SPATIAL = SAR_ALL + list(field_means.columns)

def add_spatial_neighbor_features(df, sar_cols, n_neighbors=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    
    for farm in df['farm_name'].unique():
        farm_mask = df['farm_name'] == farm
        farm_df = df[farm_mask]
        if len(farm_df) <= n_neighbors: continue
            
        tree = BallTree(farm_df[['latitude', 'longitude']].values)
        _, indices = tree.query(farm_df[['latitude', 'longitude']].values, k=n_neighbors+1)
        
        for i, col in enumerate(sar_cols):
            vals = farm_df[col].values
            neighbor_data[farm_mask, i] = vals[indices[:, 1:]].mean(axis=1)
            
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbor_features(df, SAR_ALL)

# ---------------- 4. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "FULL_SPATIAL" 

FEATURE_MAP = {
    "BASE":         SAR_ALL + ["Year", "latitude"],
    "WEATHER":      SAR_ALL + WEATHER + ["Year", "latitude"],
    "TERRAIN":      SAR_ALL + WEATHER + TERRAIN + SOIL + ["Year", "latitude"],
    "FULL_SPATIAL": SAR_SPATIAL + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["Year", "latitude", "longitude"]
}

SELECTED_FEATURES = FEATURE_MAP[RUN_MODE]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["Year"]

# Stratification for balanced folds
y_binned = pd.qcut(y, q=5, labels=False)

print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: H200 GPU")

# ---------------- 5. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]
    
    # Year-Mean Normalization
    yr_map = df.iloc[tr].groupby("Year")["yield"].mean().to_dict()
    global_mean = y_tr.mean()
    mu_tr = years.iloc[tr].map(yr_map).fillna(global_mean).values
    mu_te = years.iloc[te].map(yr_map).fillna(global_mean).values

    # Model Definition (with Early Stopping in constructor for XGB 2.0+)
    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        reg_alpha=1.0,
        early_stopping_rounds=50
    )
    
    model.fit(X_tr, y_tr / mu_tr, eval_set=[(X_te, y_te / mu_te)], verbose=False)
    
    p_ratio = model.predict(X_te)
    p_raw = p_ratio * mu_te

    # --- METRIC CALCULATIONS ---
    r2 = r2_score(y_te, p_raw)
    mae = mean_absolute_error(y_te, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te, p_raw))
    
    y_range = np.max(y_te) - np.min(y_te)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    mape = mean_absolute_percentage_error(y_te, p_raw) * 100
    
    z_true = np.where(y_te/mu_te < 0.9, 0, np.where(y_te/mu_te <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        'R2': r2, 'MAE': mae, 'RMSE': rmse, 
        'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc
    })

    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 6. DETAILED SUMMARY ----------------
res_df = pd.DataFrame(results)

print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 694250 rows with 323 columns.
Calculating Field-Level Spatial Features (Means)...
Generating BallTree neighbor features for 264 features...
🚀 Mode: FULL_SPATIAL | Features: 825 | Device: H200 GPU
Fold 1 | R2: 0.534 | RMSE: 9.19 | MAPE: 15.33% | Time: 6.7s
Fold 2 | R2: 0.453 | RMSE: 10.82 | MAPE: 24.29% | Time: 5.3s
Fold 3 | R2: 0.602 | RMSE: 8.62 | MAPE: 15.81% | Time: 6.8s
Fold 4 | R2: 0.403 | RMSE: 10.43 | MAPE: 19.62% | Time: 7.4s
Fold 5 | R2: 0.192 | RMSE: 12.15 | MAPE: 20.96% | Time: 4.1s

📊 FINAL SUMMARY: FULL_SPATIAL
Total Time: 31.04s
-----------------------------------
R²:    0.4367 ± 0.1565
MAE:   7.86
RMSE:  10.24
NRMSE: 10.83%
MAPE:  19.20%
Acc:   0.5089
-----------------------------------


# SAR features data set with weather, terrain features, spatial features, and ablations 

# Trying with ablated dataset

In [ ]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
ORIGINAL_PATH = "data/soybeans_2016_2023_processed.parquet"
ABLATED_PATH = "data/gamma_k8_stacked_ablated_corn_wheat_soy.parquet"

df_orig = pd.read_parquet(ORIGINAL_PATH)
df_orig.columns = [c.lower() for c in df_orig.columns]
df_orig = df_orig[df_orig["yield"].notna()].copy()
df_orig['is_ablation'] = False

valid_years = df_orig['year'].unique()

# Load and Filter Ablated Data
df_ablated = pd.read_parquet(ABLATED_PATH)
df_ablated.columns = [c.lower() for c in df_ablated.columns]

# FILTER: Soy only + Matching Years only
df_ablated = df_ablated[
    (df_ablated['crop'].str.contains('soy', case=False, na=False)) & 
    (df_ablated['year'].isin(valid_years))
].copy()
df_ablated['is_ablation'] = True

# Rename Ablated Farms to avoid leakage in GroupKFold
unique_farms = df_ablated['farm_name'].unique()
farm_id_map = {name: f"ablation_field_ID_{i}" for i, name in enumerate(unique_farms)}
df_ablated['farm_name'] = df_ablated['farm_name'].map(farm_id_map)

# Combine datasets
common_cols = list(set(df_orig.columns) & set(df_ablated.columns))
df = pd.concat([df_orig[common_cols + ['is_ablation']], 
                df_ablated[common_cols + ['is_ablation']]], axis=0).reset_index(drop=True)

print(f"Loaded {len(df_orig)} real rows and {len(df_ablated)} filtered ablated rows.")

# ---------------- 2. FEATURE GROUPS & SPATIAL ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("vv_") or c.startswith("vh_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("pr_", "rvi_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["dem", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]
SAR_ALL = SAR_RAW + SAR_INDICES

print("Calculating Spatial Features...")
field_means = df.groupby(['farm_name', 'year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

def add_spatial_neighbors(df, sar_cols, n=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    for farm in df['farm_name'].unique():
        mask = df['farm_name'] == farm
        f_df = df[mask]
        if len(f_df) <= n: continue
        tree = BallTree(f_df[['latitude', 'longitude']].values)
        _, idx = tree.query(f_df[['latitude', 'longitude']].values, k=n+1)
        for i, col in enumerate(sar_cols):
            neighbor_data[mask, i] = f_df[col].values[idx[:, 1:]].mean(axis=1)
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbors(df, SAR_ALL)

# ---------------- 3. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "FULL_SPATIAL_WITH_ABLATION"

SELECTED_FEATURES = SAR_ALL + list(field_means.columns) + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["year", "latitude"]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["year"].to_numpy()
is_ablation = df["is_ablation"].to_numpy().flatten()

y_binned = pd.qcut(y, q=5, labels=False)
print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: GPU")

# ---------------- 4. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    # Filter test set for REAL fields only
    te_bool_mask = (is_ablation[te] == False)
    real_te_idx = te[te_bool_mask]
    
    if len(real_te_idx) == 0: continue
    
    X_tr, X_te_real = X.iloc[tr], X.iloc[real_te_idx]
    y_te_real = y[real_te_idx]
    
    # Normalization factors
    yr_map = df.iloc[tr].groupby("year")["yield"].mean().to_dict()
    global_mean = y[tr].mean()
    mu_tr = pd.Series(years[tr]).map(yr_map).fillna(global_mean).values
    mu_te_real = pd.Series(years[real_te_idx]).map(yr_map).fillna(global_mean).values

    model = xgb.XGBRegressor(
        tree_method="hist", device="cuda", n_estimators=1000, max_depth=6,
        learning_rate=0.03, subsample=0.7, colsample_bytree=0.6,
        n_jobs=CPUS, random_state=42, reg_lambda=10.0, early_stopping_rounds=50
    )
    
    model.fit(X_tr, y[tr] / mu_tr, eval_set=[(X_te_real, y_te_real / mu_te_real)], verbose=False)
    
    p_ratio = model.predict(X_te_real)
    p_raw = p_ratio * mu_te_real

    # Metrics
    r2 = r2_score(y_te_real, p_raw)
    mae = mean_absolute_error(y_te_real, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te_real, p_raw))
    mape = mean_absolute_percentage_error(y_te_real, p_raw) * 100
    y_range = np.max(y_te_real) - np.min(y_te_real)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    
    z_true = np.where(y_te_real/mu_te_real < 0.9, 0, np.where(y_te_real/mu_te_real <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({'R2': r2, 'MAE': mae, 'RMSE': rmse, 'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc})
    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 5. FINAL SUMMARY ----------------
res_df = pd.DataFrame(results)
print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)

# SAR features data set with weather, terrain features, spatial features, and ALL ablations

In [ ]:
import os, time, gc
import numpy as np
import torch
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
ORIGINAL_PATH = "data/soybeans_2016_2023_processed.parquet"
ABLATED_PATH = "data/gamma_k8_stacked_ablated_TERRAIN_WEATHER_corn_wheat_soy.parquet"

# Load and filter immediately to save RAM
df_orig = pd.read_parquet(ORIGINAL_PATH)
df_orig.columns = [c.lower() for c in df_orig.columns]
df_orig = df_orig[df_orig["yield"].notna()].copy()
df_orig["is_ablation"] = False
valid_years = df_orig["year"].unique()

df_ablated = pd.read_parquet(ABLATED_PATH)
df_ablated.columns = [c.lower() for c in df_ablated.columns]
df_ablated = df_ablated[
    (df_ablated["crop"].str.contains("soy", case=False, na=False)) &
    (df_ablated["year"].isin(valid_years))
].copy()
df_ablated["is_ablation"] = True

# Rename Ablated Farms
unique_farms = df_ablated["farm_name"].unique()
farm_id_map = {name: f"ablation_field_id_{i}" for i, name in enumerate(unique_farms)}
df_ablated["farm_name"] = df_ablated["farm_name"].map(farm_id_map)

# Align columns and concatenate
all_cols = sorted(set(df_orig.columns) | set(df_ablated.columns))
df = pd.concat([
    df_orig.reindex(columns=all_cols), 
    df_ablated.reindex(columns=all_cols)
], axis=0, ignore_index=True)

print(f"Loaded {len(df_orig)} real and {len(df_ablated)} ablated rows.")
del df_orig, df_ablated # CLEAR IMMEDIATELY
gc.collect()

# ---------------- 2. FEATURE GROUPS & SPATIAL ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("vv_") or c.startswith("vh_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("pr_", "rvi_"))]
SAR_ALL = SAR_RAW + SAR_INDICES
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"])]
TERRAIN = ["dem", "slope_deg", "aspect_deg", "shape_index"]
SOIL = [c for c in df.columns if c.startswith("soil_")]

print("Calculating Spatial Features...")
# Use transform to keep index alignment, cast to float32
field_means = df.groupby(["farm_name", "year"])[SAR_ALL].transform("mean").astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

def add_spatial_neighbors_fast(df, cols, n=8, only_real=True):
    print(f"Generating BallTree features for {len(cols)} columns...")
    out = np.full((len(df), len(cols)), np.nan, dtype=np.float32)
    base_mask = ~df["is_ablation"].astype(bool) if only_real else pd.Series(True, index=df.index)

    for farm, idx in df.loc[base_mask].groupby("farm_name").groups.items():
        idx = np.array(list(idx), dtype=int)
        if idx.size <= n: continue
        
        f_df = df.loc[idx]
        coords = f_df[["latitude", "longitude"]].to_numpy(dtype=np.float64)
        tree = BallTree(coords)
        _, nn = tree.query(coords, k=n+1)
        nn = nn[:, 1:]
        
        A = f_df[cols].to_numpy(dtype=np.float32)
        neigh = A[nn]
        s = np.nansum(neigh, axis=1)
        c = np.sum(~np.isnan(neigh), axis=1)
        m = s / np.maximum(c, 1)
        m[c == 0] = np.nan
        out[idx, :] = m

    new_cols = [f"{c}_neighbor_{n}" for c in cols]
    df = pd.concat([df, pd.DataFrame(out, columns=new_cols, index=df.index)], axis=1)
    return df, new_cols

df, SAR_NEIGHBORS = add_spatial_neighbors_fast(df, SAR_ALL, n=8, only_real=True)

# ---------------- 3. CONFIGURATION ----------------
SELECTED_FEATURES = SAR_ALL + list(field_means.columns) + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["year", "latitude"]
SELECTED_FEATURES = [c for c in SELECTED_FEATURES if c in df.columns]

X = df[SELECTED_FEATURES].astype(np.float32)
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["year"].to_numpy(dtype=np.int32)
is_ablation = df["is_ablation"].to_numpy().flatten()

del df, field_means # FINAL DATA CLEANUP
gc.collect()

y_binned = pd.qcut(y, q=5, labels=False, duplicates="drop")
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))

# ---------------- 4. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    real_te_idx = te[is_ablation[te] == False]
    if len(real_te_idx) == 0: continue

    X_tr, X_te_real = X.iloc[tr], X.iloc[real_te_idx]
    y_tr, y_te_real = y[tr], y[real_te_idx]

    unique_yrs = np.unique(years[tr])
    yr_map = {yr: y_tr[years[tr] == yr].mean() for yr in unique_yrs}
    mu_tr = np.array([yr_map.get(yr, y_tr.mean()) for yr in years[tr]])
    mu_te_real = np.array([yr_map.get(yr, y_tr.mean()) for yr in years[real_te_idx]])

    model = xgb.XGBRegressor(
        tree_method="hist", device="cuda", n_estimators=1000, max_depth=6,
        learning_rate=0.03, subsample=0.7, colsample_bytree=0.6, n_jobs=CPUS,
        random_state=42, reg_lambda=10.0, early_stopping_rounds=50
    )

    model.fit(X_tr, y_tr/mu_tr, eval_set=[(X_te_real, y_te_real/mu_te_real)], verbose=False)

    p_ratio = model.predict(X_te_real)
    p_raw = p_ratio * mu_te_real

    # Metrics
    r2 = r2_score(y_te_real, p_raw)
    mae = mean_absolute_error(y_te_real, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te_real, p_raw))
    mape = mean_absolute_percentage_error(y_te_real, p_raw) * 100
    y_range = np.max(y_te_real) - np.min(y_te_real)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    z_true = np.where(y_te_real/mu_te_real < 0.9, 0, np.where(y_te_real/mu_te_real <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({"R2": r2, "MAE": mae, "RMSE": rmse, "NRMSE": nrmse, "MAPE": mape, "Zone_Acc": acc})
    print(f"Fold {k} | R2: {r2:.3f} | MAE: {mae:.2f} | RMSE: {rmse:.2f} | Acc: {acc:.4f} | Time: {time.time()-f_start:.1f}s")
    
    del X_tr, X_te_real, model
    gc.collect()
    torch.cuda.empty_cache()

# ---------------- 5. FINAL SUMMARY ----------------
res_df = pd.DataFrame(results)
print(f"\n📊 FINAL SUMMARY")
print("-" * 35)
for m in ["R2", "MAE", "RMSE", "NRMSE", "MAPE", "Zone_Acc"]:
    print(f"{m:8}: {res_df[m].mean():.4f} ± {res_df[m].std():.4f}")
print("-" * 35)

# Making the prediction column

In [1]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
ORIGINAL_PATH = "data/soybeans_2016_2023_processed.parquet"
ABLATED_PATH  = "data/gamma_k8_stacked_ablated_corn_wheat_soy.parquet"

OUTPUT_PATH   = "data/soybeans_full_spatial_with_ablation__with_oof_preds.parquet"
PRED_COL      = "pred_yield_oof"
PRED_FOLD_COL = "pred_fold"

TARGET_CROP = "soybean"  # ✅ IMPORTANT

df_orig = pd.read_parquet(ORIGINAL_PATH)
df_orig.columns = [c.lower() for c in df_orig.columns]
df_orig = df_orig[df_orig["yield"].notna()].copy()
df_orig["is_ablation"] = False

valid_years = df_orig["year"].unique()

# Load and Filter Ablated Data
df_ablated = pd.read_parquet(ABLATED_PATH)
df_ablated.columns = [c.lower() for c in df_ablated.columns]

# ✅ FILTER: CORN only + Matching Years only
df_ablated = df_ablated[
    (df_ablated["crop"].str.contains(TARGET_CROP, case=False, na=False)) &
    (df_ablated["year"].isin(valid_years))
].copy()
df_ablated["is_ablation"] = True

# Rename Ablated Farms to avoid leakage in GroupKFold
unique_farms = df_ablated["farm_name"].unique()
farm_id_map = {name: f"ablation_field_ID_{i}" for i, name in enumerate(unique_farms)}
df_ablated["farm_name"] = df_ablated["farm_name"].map(farm_id_map)

# Combine datasets (deterministic column order + no duplicate is_ablation)
common_cols = [c for c in df_orig.columns if c in df_ablated.columns]
# Ensure is_ablation appears exactly once and at the end (nice for readability)
common_cols = [c for c in common_cols if c != "is_ablation"] + ["is_ablation"]

df = pd.concat(
    [df_orig[common_cols], df_ablated[common_cols]],
    axis=0
).reset_index(drop=True)

# Safety: drop any accidental duplicates (should be none now)
df = df.loc[:, ~df.columns.duplicated()]

print(f"Loaded {len(df_orig)} real rows and {len(df_ablated)} filtered ablated rows for crop='{TARGET_CROP}'.")

# ---------------- 2. FEATURE GROUPS & SPATIAL ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("vv_") or c.startswith("vh_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("pr_", "rvi_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["dem", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]
SAR_ALL = SAR_RAW + SAR_INDICES

print("Calculating Spatial Features...")
field_means = df.groupby(["farm_name", "year"])[SAR_ALL].transform("mean").astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

def add_spatial_neighbors(df_in, sar_cols, n=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df_in), len(sar_cols)), dtype=np.float32)

    for farm in df_in["farm_name"].unique():
        mask = (df_in["farm_name"] == farm).to_numpy()
        f_df = df_in.loc[mask, :]
        if len(f_df) <= n:
            continue

        coords = f_df[["latitude", "longitude"]].to_numpy()
        tree = BallTree(coords)
        _, idx = tree.query(coords, k=n + 1)  # includes self at idx[:,0]

        for i, col in enumerate(sar_cols):
            vals = f_df[col].to_numpy()
            neighbor_data[mask, i] = vals[idx[:, 1:]].mean(axis=1)

    new_cols = [f"{c}_neighbor_{n}" for c in sar_cols]
    df_out = pd.concat([df_in, pd.DataFrame(neighbor_data, columns=new_cols, index=df_in.index)], axis=1)
    return df_out, new_cols

df, SAR_NEIGHBORS = add_spatial_neighbors(df, SAR_ALL, n=8)

# ---------------- 3. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "FULL_SPATIAL_WITH_ABLATION"

SELECTED_FEATURES = SAR_ALL + list(field_means.columns) + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["year", "latitude"]

X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["year"].to_numpy()
is_ablation = df["is_ablation"].to_numpy().flatten()

y_binned = pd.qcut(y, q=5, labels=False)
print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: GPU")

# Allocate prediction columns
df[PRED_COL] = np.nan
df[PRED_FOLD_COL] = np.nan

# ---------------- 4. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()

    # Filter test set for REAL fields only
    te_bool_mask = (is_ablation[te] == False)
    real_te_idx = te[te_bool_mask]
    if len(real_te_idx) == 0:
        continue

    X_tr, X_te_real = X.iloc[tr], X.iloc[real_te_idx]
    y_te_real = y[real_te_idx]

    # Normalization factors
    yr_map = df.iloc[tr].groupby("year")["yield"].mean().to_dict()
    global_mean = float(y[tr].mean())
    mu_tr = pd.Series(years[tr]).map(yr_map).fillna(global_mean).values
    mu_te_real = pd.Series(years[real_te_idx]).map(yr_map).fillna(global_mean).values

    model = xgb.XGBRegressor(
        tree_method="hist", device="cuda", n_estimators=1000, max_depth=6,
        learning_rate=0.03, subsample=0.7, colsample_bytree=0.6,
        n_jobs=CPUS, random_state=42, reg_lambda=10.0, early_stopping_rounds=50
    )

    model.fit(X_tr, y[tr] / mu_tr, eval_set=[(X_te_real, y_te_real / mu_te_real)], verbose=False)

    p_ratio = model.predict(X_te_real)
    p_raw = p_ratio * mu_te_real

    # Store OOF predictions
    df.loc[real_te_idx, PRED_COL] = p_raw.astype(np.float32)
    df.loc[real_te_idx, PRED_FOLD_COL] = k

    # Metrics
    r2 = r2_score(y_te_real, p_raw)
    mae = mean_absolute_error(y_te_real, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te_real, p_raw))
    mape = mean_absolute_percentage_error(y_te_real, p_raw) * 100
    y_range = np.max(y_te_real) - np.min(y_te_real)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0

    z_true = np.where(y_te_real/mu_te_real < 0.9, 0, np.where(y_te_real/mu_te_real <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({'R2': r2, 'MAE': mae, 'RMSE': rmse, 'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc})
    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 5. FINAL SUMMARY ----------------
res_df = pd.DataFrame(results)
print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)

# ---------------- 6. SAVE DATASET WITH PREDICTIONS ----------------
df = df.loc[:, ~df.columns.duplicated()]
df.to_parquet(OUTPUT_PATH, index=False)
print(f"\n✅ Saved dataframe (with OOF predictions) to: {OUTPUT_PATH}")
print(f"Columns added: {PRED_COL}, {PRED_FOLD_COL}")

Loaded 694250 real rows and 1307054 filtered ablated rows for crop='soybean'.
Calculating Spatial Features...
Generating BallTree neighbor features for 132 features...
🚀 Mode: FULL_SPATIAL_WITH_ABLATION | Features: 428 | Device: GPU
Fold 1 | R2: 0.635 | RMSE: 8.93 | MAPE: 15.01% | Time: 14.9s
Fold 2 | R2: 0.807 | RMSE: 6.13 | MAPE: 11.30% | Time: 15.1s
Fold 3 | R2: 0.724 | RMSE: 6.60 | MAPE: 10.94% | Time: 15.2s
Fold 4 | R2: 0.631 | RMSE: 8.07 | MAPE: 12.71% | Time: 15.4s
Fold 5 | R2: 0.674 | RMSE: 7.91 | MAPE: 15.03% | Time: 13.8s

📊 FINAL SUMMARY: FULL_SPATIAL_WITH_ABLATION
Total Time: 76.86s
-----------------------------------
R²:    0.6941 ± 0.0734
MAE:   5.62
RMSE:  7.53
NRMSE: 8.07%
MAPE:  13.00%
Acc:   0.6609
-----------------------------------

✅ Saved dataframe (with OOF predictions) to: data/soybeans_full_spatial_with_ablation__with_oof_preds.parquet
Columns added: pred_yield_oof, pred_fold


In [2]:
print("Ablated crop values (top):", df_ablated["crop"].value_counts().head(10))

Ablated crop values (top): crop
soybeans    1307054
Name: count, dtype: int64


In [ ]:
# 